# 10 — Evaluation and metadata


In [18]:
from pathlib import Path
import datetime
import json
import platform
import sys

import geopandas
import osmnx
import pandas as pd
import rdflib
import sklearn

# ---------------------------------------------------------
# Resolve project root
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/subhankarbiswas/smart-city-knowledge-graph-v3


In [19]:
from src.config import settings

print("=" * 60)
print("PROJECT CONFIGURATION")
print("=" * 60)

print(f"City:                 {settings.city_name}")
print(f"Walking speed:        {settings.walking_speed_kph} km/h")
print(f"Max accessibility origins: {settings.accessibility_max_origins}")

PROJECT CONFIGURATION
City:                 Eschwege, Hesse, Germany
Walking speed:        5.0 km/h
Max accessibility origins: 750


In [20]:
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
GRAPH_DIR = OUTPUT_DIR / "graphs"
MAP_DIR = OUTPUT_DIR / "maps"

METADATA_PATH = (
    PROCESSED_DIR / "run_metadata.json"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Raw data:       {RAW_DIR}")
print(f"Processed data: {PROCESSED_DIR}")
print(f"Graphs:         {GRAPH_DIR}")
print(f"Maps:           {MAP_DIR}")

Raw data:       /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/raw
Processed data: /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed
Graphs:         /Users/subhankarbiswas/smart-city-knowledge-graph-v3/outputs/graphs
Maps:           /Users/subhankarbiswas/smart-city-knowledge-graph-v3/outputs/maps


In [21]:
print("=" * 60)
print("GENERATED DATASETS")
print("=" * 60)

dataset_paths = {
    "POIs": PROCESSED_DIR / "pois_clean.parquet",
    "POIs + Wikidata": PROCESSED_DIR / "pois_wikidata.parquet",
    "Buildings": PROCESSED_DIR / "buildings_clean.parquet",
    "Roads": PROCESSED_DIR / "roads_clean.parquet",
    "Transport": PROCESSED_DIR / "transport_clean.parquet",
    "Network accessibility": (
        PROCESSED_DIR / "network_accessibility.csv"
    ),
    "Accessibility coverage": (
        PROCESSED_DIR / "accessibility_coverage.csv"
    ),
    "Accessibility profiles": (
        PROCESSED_DIR / "accessibility_profiles.csv"
    ),
}

dataset_status = []

for name, path in dataset_paths.items():

    exists = path.exists()

    row = {
        "dataset": name,
        "exists": exists,
        "path": str(path),
        "size_kb": (
            round(path.stat().st_size / 1024, 2)
            if exists
            else None
        ),
    }

    dataset_status.append(row)

dataset_status_df = pd.DataFrame(dataset_status)

display(dataset_status_df)

GENERATED DATASETS


,dataset,exists,path,size_kb
0,POIs,True,/Users/subhankarbiswas/smart-city-knowledge-gr...,67.89
1,POIs + Wikidata,True,/Users/subhankarbiswas/smart-city-knowledge-gr...,73.33
2,Buildings,True,/Users/subhankarbiswas/smart-city-knowledge-gr...,1032.98
3,Roads,True,/Users/subhankarbiswas/smart-city-knowledge-gr...,244.80
4,Transport,True,/Users/subhankarbiswas/smart-city-knowledge-gr...,34.07
5,Network accessibility,True,/Users/subhankarbiswas/smart-city-knowledge-gr...,6.31
6,Accessibility coverage,True,/Users/subhankarbiswas/smart-city-knowledge-gr...,0.55
7,Accessibility profiles,True,/Users/subhankarbiswas/smart-city-knowledge-gr...,1.93


In [22]:
dataset_statistics = []

for name, path in dataset_paths.items():

    if not path.exists():
        continue

    try:

        if path.suffix == ".parquet":
            gdf = geopandas.read_parquet(path)

            dataset_statistics.append({
                "dataset": name,
                "rows": len(gdf),
                "columns": len(gdf.columns),
                "crs": str(gdf.crs),
            })

        elif path.suffix == ".csv":
            df = pd.read_csv(path)

            dataset_statistics.append({
                "dataset": name,
                "rows": len(df),
                "columns": len(df.columns),
                "crs": None,
            })

    except Exception as exc:

        print(
            f"Could not inspect {name}: {exc}"
        )

dataset_statistics_df = pd.DataFrame(
    dataset_statistics
)

print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

display(dataset_statistics_df)

DATASET STATISTICS


,dataset,rows,columns,crs
0,POIs,224,88,"{""$schema"": ""https://proj.org/schemas/v0.7/pro..."
1,POIs + Wikidata,225,96,"{""$schema"": ""https://proj.org/schemas/v0.7/pro..."
2,Buildings,10347,173,"{""$schema"": ""https://proj.org/schemas/v0.7/pro..."
3,Roads,2355,19,"{""$schema"": ""https://proj.org/schemas/v0.7/pro..."
4,Transport,165,39,"{""$schema"": ""https://proj.org/schemas/v0.7/pro..."
5,Network accessibility,110,5,NaN
6,Accessibility coverage,15,5,NaN
7,Accessibility profiles,22,6,NaN


In [23]:
POI_PATH = PROCESSED_DIR / "pois_clean.parquet"

poi_statistics = {}

if POI_PATH.exists():

    pois = geopandas.read_parquet(
        POI_PATH
    )

    poi_statistics = {
        "total_pois": int(len(pois)),
        "unique_service_types": (
            int(
                pois["service_type"]
                .nunique()
            )
            if "service_type" in pois.columns
            else None
        ),
    }

    if "service_type" in pois.columns:

        service_distribution = (
            pois["service_type"]
            .fillna("unknown")
            .value_counts()
            .to_dict()
        )

        poi_statistics[
            "service_type_distribution"
        ] = {
            str(k): int(v)
            for k, v in service_distribution.items()
        }

print("=" * 60)
print("POI EVALUATION")
print("=" * 60)

print(
    json.dumps(
        poi_statistics,
        indent=2
    )
)

POI EVALUATION
{
  "total_pois": 224,
  "unique_service_types": 10,
  "service_type_distribution": {
    "bus_stop": 164,
    "park": 20,
    "school": 12,
    "supermarket": 10,
    "atm": 6,
    "pharmacy": 5,
    "bank": 4,
    "hospital": 1,
    "train_station": 1,
    "library": 1
  }
}


In [24]:
GRAPH_PATH = (
    GRAPH_DIR / "smart_city_kg.ttl"
)

kg_statistics = {}

if GRAPH_PATH.exists():

    graph = rdflib.Graph()

    graph.parse(
        GRAPH_PATH,
        format="turtle"
    )

    kg_statistics = {
        "rdf_triples": int(len(graph)),
    }

    print("=" * 60)
    print("KNOWLEDGE GRAPH")
    print("=" * 60)

    print(
        f"RDF triples: {len(graph):,}"
    )

else:

    print(
        "Knowledge graph not found:"
    )
    print(GRAPH_PATH)

KNOWLEDGE GRAPH
RDF triples: 49,930


In [25]:
if GRAPH_PATH.exists():
    
    entity_type_query = """
    SELECT ?type (COUNT(?entity) AS ?count)
    WHERE {
        ?entity a ?type .
    }
    GROUP BY ?type
    ORDER BY DESC(?count)
    """

    results = graph.query(
        entity_type_query
    )

    rows = [
        (
            str(row.type),
            int(row["count"])
        )
        for row in results
    ]

    kg_entity_types = pd.DataFrame(
        rows,
        columns=[
            "rdf_type",
            "count",
        ],
    )

    display(kg_entity_types)

else:

    kg_entity_types = pd.DataFrame()

,rdf_type,count
0,http://www.opengis.net/ont/geosparql#Geometry,11683
1,https://example.org/smartcity/Building,10347
2,https://example.org/smartcity/Road,947
3,https://example.org/smartcity/POI,224
4,https://example.org/smartcity/Transport,165


In [26]:
if GRAPH_PATH.exists():
    
    wikidata_query = """
    SELECT (COUNT(?entity) AS ?count)
    WHERE {
        ?entity
            <http://www.w3.org/2002/07/owl#sameAs>
            ?wikidata .
    }
    """

    result = list(
        graph.query(wikidata_query)
    )

    wikidata_links = (
        int(result[0]["count"])
        if result
        else 0
    )

else:

    wikidata_links = 0

print(
    f"Wikidata reconciliation links: "
    f"{wikidata_links:,}"
)

Wikidata reconciliation links: 5


In [27]:
ACCESSIBILITY_PATH = (
    PROCESSED_DIR / "network_accessibility.csv"
)

accessibility_statistics = {}

if ACCESSIBILITY_PATH.exists():

    accessibility = pd.read_csv(
        ACCESSIBILITY_PATH
    )

    accessibility_statistics = {
        "records": int(len(accessibility)),
        "services": (
            int(
                accessibility["service"]
                .nunique()
            )
            if "service" in accessibility.columns
            else None
        ),
        "origins": (
            int(
                accessibility["origin_index"]
                .nunique()
            )
            if "origin_index"
            in accessibility.columns
            else None
        ),
    }

    if (
        "travel_time_min"
        in accessibility.columns
    ):

        travel_times = pd.to_numeric(
            accessibility[
                "travel_time_min"
            ],
            errors="coerce",
        ).dropna()

        if not travel_times.empty:

            accessibility_statistics.update({
                "travel_time_mean_min": round(
                    float(travel_times.mean()),
                    2,
                ),
                "travel_time_median_min": round(
                    float(travel_times.median()),
                    2,
                ),
                "travel_time_min": round(
                    float(travel_times.min()),
                    2,
                ),
                "travel_time_max": round(
                    float(travel_times.max()),
                    2,
                ),
            })

print("=" * 60)
print("NETWORK ACCESSIBILITY")
print("=" * 60)

print(
    json.dumps(
        accessibility_statistics,
        indent=2
    )
)

NETWORK ACCESSIBILITY
{
  "records": 110,
  "services": 5,
  "origins": 22,
  "travel_time_mean_min": 16.2,
  "travel_time_median_min": 13.58,
  "travel_time_min": 0.0,
  "travel_time_max": 47.57
}


In [28]:
COVERAGE_PATH = (
    PROCESSED_DIR
    / "accessibility_coverage.csv"
)

coverage_statistics = {}

if COVERAGE_PATH.exists():

    coverage = pd.read_csv(
        COVERAGE_PATH
    )

    coverage_statistics = {
        "records": int(len(coverage)),
        "services": (
            int(
                coverage["service"]
                .nunique()
            )
            if "service" in coverage.columns
            else None
        ),
    }

print("=" * 60)
print("ACCESSIBILITY COVERAGE")
print("=" * 60)

print(
    json.dumps(
        coverage_statistics,
        indent=2
    )
)

ACCESSIBILITY COVERAGE
{
  "records": 15,
  "services": 5
}


In [29]:
KMEANS_PATH = (
    PROCESSED_DIR
    / "accessibility_profiles.csv"
)

DBSCAN_PATH = (
    PROCESSED_DIR
    / "accessibility_profiles_dbscan.csv"
)

clustering_statistics = {}

if KMEANS_PATH.exists():

    kmeans = pd.read_csv(
        KMEANS_PATH
    )

    if "cluster_kmeans" in kmeans.columns:

        clustering_statistics[
            "kmeans_origins"
        ] = int(len(kmeans))

        clustering_statistics[
            "kmeans_clusters"
        ] = int(
            kmeans["cluster_kmeans"]
            .nunique()
        )

        clustering_statistics[
            "kmeans_distribution"
        ] = {
            str(k): int(v)
            for k, v in (
                kmeans["cluster_kmeans"]
                .value_counts()
                .to_dict()
                .items()
            )
        }


if DBSCAN_PATH.exists():

    dbscan = pd.read_csv(
        DBSCAN_PATH
    )

    if "cluster_dbscan" in dbscan.columns:

        clustering_statistics[
            "dbscan_origins"
        ] = int(len(dbscan))

        clustering_statistics[
            "dbscan_clusters"
        ] = int(
            dbscan["cluster_dbscan"]
            .nunique()
        )

        clustering_statistics[
            "dbscan_distribution"
        ] = {
            str(k): int(v)
            for k, v in (
                dbscan["cluster_dbscan"]
                .value_counts()
                .to_dict()
                .items()
            )
        }

print("=" * 60)
print("CLUSTERING EVALUATION")
print("=" * 60)

print(
    json.dumps(
        clustering_statistics,
        indent=2
    )
)

CLUSTERING EVALUATION
{
  "kmeans_origins": 22,
  "kmeans_clusters": 4,
  "kmeans_distribution": {
    "1": 7,
    "2": 6,
    "3": 6,
    "0": 3
  },
  "dbscan_origins": 22,
  "dbscan_clusters": 1,
  "dbscan_distribution": {
    "-1": 22
  }
}


In [30]:
environment_metadata = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "architecture": platform.machine(),
    "osmnx": osmnx.__version__,
    "geopandas": geopandas.__version__,
    "rdflib": rdflib.__version__,
    "scikit_learn": sklearn.__version__,
    "pandas": pd.__version__,
}

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)

display(
    pd.DataFrame(
        [
            {
                "package": key,
                "version": value,
            }
            for key, value in environment_metadata.items()
        ]
    )
)

ENVIRONMENT


,package,version
0,python,3.12.7
1,platform,macOS-26.6.2-arm64-arm-64bit
2,architecture,arm64
3,osmnx,2.1.1
4,geopandas,1.1.4
5,rdflib,7.6.0
6,scikit_learn,1.9.1
7,pandas,3.0.6


In [31]:
timestamp_utc = (
    datetime.datetime
    .now(datetime.timezone.utc)
    .isoformat()
)

metadata = {
    "project": {
        "name": "Smart City Knowledge Graph & Network Accessibility Analyzer",
        "city": settings.city_name,
        "timestamp_utc": timestamp_utc,
    },

    "configuration": {
        "walking_speed_kph": settings.walking_speed_kph,
        "accessibility_max_origins": (
            settings.accessibility_max_origins
        ),
    },

    "environment": environment_metadata,

    "datasets": dataset_statistics,

    "poi_analysis": poi_statistics,

    "knowledge_graph": {
        **kg_statistics,
        "wikidata_links": wikidata_links,
        "entity_types": (
            kg_entity_types.to_dict(
                orient="records"
            )
            if not kg_entity_types.empty
            else []
        ),
    },

    "network_accessibility": (
        accessibility_statistics
    ),

    "accessibility_coverage": (
        coverage_statistics
    ),

    "clustering": (
        clustering_statistics
    ),
}

print("=" * 60)
print("RUN METADATA")
print("=" * 60)

print(
    json.dumps(
        metadata,
        indent=2,
        default=str,
    )
)

RUN METADATA
{
  "project": {
    "name": "Smart City Knowledge Graph & Network Accessibility Analyzer",
    "city": "Eschwege, Hesse, Germany",
    "timestamp_utc": "2026-09-20T01:42:21.833049+00:00"
  },
  "configuration": {
    "walking_speed_kph": 5.0,
    "accessibility_max_origins": 750
  },
  "environment": {
    "python": "3.12.7",
    "platform": "macOS-26.6.2-arm64-arm-64bit",
    "architecture": "arm64",
    "osmnx": "2.1.1",
    "geopandas": "1.1.4",
    "rdflib": "7.6.0",
    "scikit_learn": "1.9.1",
    "pandas": "3.0.6"
  },
  "datasets": [
    {
      "dataset": "POIs",
      "rows": 224,
      "columns": 88,
      "crs": "{\"$schema\": \"https://proj.org/schemas/v0.7/projjson.schema.json\", \"type\": \"GeographicCRS\", \"name\": \"WGS 84\", \"datum_ensemble\": {\"name\": \"World Geodetic System 1984 ensemble\", \"members\": [{\"name\": \"World Geodetic System 1984 (Transit)\"}, {\"name\": \"World Geodetic System 1984 (G730)\"}, {\"name\": \"World Geodetic System 1984 (

In [32]:
METADATA_PATH.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

print("✓ Run metadata saved:")
print(METADATA_PATH)

✓ Run metadata saved:
/Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/run_metadata.json


In [33]:
print("=" * 60)
print("FINAL PROJECT EVALUATION")
print("=" * 60)

print(
    f"""
Project
-------
Smart City Knowledge Graph & Network Accessibility Analyzer

City
----
{settings.city_name}

Knowledge Graph
---------------
RDF triples:              {kg_statistics.get("rdf_triples", 0):,}
Wikidata links:           {wikidata_links:,}

POIs
----
Total POIs:               {poi_statistics.get("total_pois", 0):,}
Service types:            {poi_statistics.get("unique_service_types", 0):,}

Network Accessibility
---------------------
Records:                  {accessibility_statistics.get("records", 0):,}
Origins:                  {accessibility_statistics.get("origins", 0):,}
Services:                 {accessibility_statistics.get("services", 0):,}

Clustering
----------
K-Means clusters:         {clustering_statistics.get("kmeans_clusters", "N/A")}
DBSCAN clusters:          {clustering_statistics.get("dbscan_clusters", "N/A")}

Environment
-----------
Python:                   {environment_metadata["python"]}
OSMnx:                    {environment_metadata["osmnx"]}
GeoPandas:                {environment_metadata["geopandas"]}
RDFLib:                   {environment_metadata["rdflib"]}
Scikit-learn:             {environment_metadata["scikit_learn"]}

Metadata
--------
{METADATA_PATH}
"""
)

FINAL PROJECT EVALUATION

Project
-------
Smart City Knowledge Graph & Network Accessibility Analyzer

City
----
Eschwege, Hesse, Germany

Knowledge Graph
---------------
RDF triples:              49,930
Wikidata links:           5

POIs
----
Total POIs:               224
Service types:            10

Network Accessibility
---------------------
Records:                  110
Origins:                  22
Services:                 5

Clustering
----------
K-Means clusters:         4
DBSCAN clusters:          1

Environment
-----------
Python:                   3.12.7
OSMnx:                    2.1.1
GeoPandas:                1.1.4
RDFLib:                   7.6.0
Scikit-learn:             1.9.1

Metadata
--------
/Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/run_metadata.json



In [34]:
print("=" * 60)
print("FINAL ARTIFACT CHECK")
print("=" * 60)

artifacts = [
    PROCESSED_DIR / "pois_clean.parquet",
    PROCESSED_DIR / "buildings_clean.parquet",
    PROCESSED_DIR / "roads_clean.parquet",
    PROCESSED_DIR / "transport_clean.parquet",
    PROCESSED_DIR / "network_accessibility.csv",
    PROCESSED_DIR / "accessibility_coverage.csv",
    PROCESSED_DIR / "accessibility_profiles.csv",
    GRAPH_DIR / "smart_city_kg.ttl",
    GRAPH_DIR / "smart_city_kg.jsonld",
    MAP_DIR / "poi_map.html",
    MAP_DIR / "travel_time_distribution.html",
    METADATA_PATH,
]

for artifact in artifacts:

    if artifact.exists():

        size_kb = (
            artifact.stat().st_size / 1024
        )

        print(
            f"✓ {artifact.relative_to(PROJECT_ROOT)} "
            f"({size_kb:,.1f} KB)"
        )

    else:

        print(
            f"✗ {artifact.relative_to(PROJECT_ROOT)} "
            f"[MISSING]"
        )

FINAL ARTIFACT CHECK
✓ data/processed/pois_clean.parquet (67.9 KB)
✓ data/processed/buildings_clean.parquet (1,033.0 KB)
✓ data/processed/roads_clean.parquet (244.8 KB)
✓ data/processed/transport_clean.parquet (34.1 KB)
✓ data/processed/network_accessibility.csv (6.3 KB)
✓ data/processed/accessibility_coverage.csv (0.5 KB)
✓ data/processed/accessibility_profiles.csv (1.9 KB)
✓ outputs/graphs/smart_city_kg.ttl (4,624.0 KB)
✓ outputs/graphs/smart_city_kg.jsonld (9,179.4 KB)
✓ outputs/maps/poi_map.html (251.9 KB)
✓ outputs/maps/travel_time_distribution.html (13.7 KB)
✓ data/processed/run_metadata.json (9.3 KB)
